# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` â€” folds, track preparation, metrics, the loop itself
- `src/models/` â€” model functions and `MODEL_REGISTRY`
- `src/features/quantum/` â€” quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: C:\dev\projects\capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "Jack"  # your name, recorded on every result row (default: OS user)
RESUME        = True  # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce",
                  "Q_heisenberg_pauli", "Q_heisenberg_single_bit", "Q_zz_pauli", "Q_zz_pce", "Q_zz_single_bit"]
ACTIVE_TRACKS = ["A"]                                          # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["decision_tree", "adaboost", "random_forest", "xgboost",
                 "linear_svm", "sig_kernel", "linear_readout",
                 "rnn", "gru", "lstm"]  # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['A']
Models:   ['decision_tree', 'adaboost', 'random_forest', 'xgboost', 'linear_svm', 'sig_kernel', 'linear_readout', 'rnn', 'gru', 'lstm']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track A        target INDPRO     loaded   shape=(767, 123)
  Track A        target PAYEMS     loaded   shape=(767, 123)
  Track A        target CPIAUCSL   loaded   shape=(767, 123)
  Track A        target S&P 500    loaded   shape=(767, 123)
Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Track A: resuming, 30896 runs already on disk will be skipped


  31896/77640 runs complete · 1m38s elapsed · ~1h14m left


  Saved: raw/linear_svm/A.parquet   new=300  total=300
  Saved: raw/sig_kernel/A.parquet   new=300  total=300
  Saved: raw/linear_readout/A.parquet   new=300  total=300
  Saved: raw/rnn/A.parquet   new=300  total=300
  Saved: raw/gru/A.parquet   new=300  total=300
  Saved: raw/lstm/A.parquet   new=300  total=300


  32896/77640 runs complete · 3m51s elapsed · ~1h26m left


  33896/77640 runs complete · 7m46s elapsed · ~1h53m left


...

  74896/77640 runs complete · 1h16m elapsed · ~4m47s left


  75896/77640 runs complete · 1h18m elapsed · ~3m01s left
  Saved: raw/linear_svm/A.parquet   new=300  total=7500
  Saved: raw/sig_kernel/A.parquet   new=300  total=7500
  Saved: raw/linear_readout/A.parquet   new=300  total=7500
  Saved: raw/rnn/A.parquet   new=300  total=7500
  Saved: raw/gru/A.parquet   new=300  total=7500
  Saved: raw/lstm/A.parquet   new=300  total=7500


  76896/77640 runs complete · 1h19m elapsed · ~1m17s left


  Saved: raw/linear_svm/A.parquet   new=224  total=7724
  Saved: raw/sig_kernel/A.parquet   new=224  total=7724
  Saved: raw/linear_readout/A.parquet   new=224  total=7724
  Saved: raw/rnn/A.parquet   new=224  total=7724
  Saved: raw/gru/A.parquet   new=224  total=7724
  Saved: raw/lstm/A.parquet   new=224  total=7724
Loop complete. Total rows: 46344 in 1h19m


Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae      rmse  \
model    track target   horizon                                            
adaboost A     CPIAUCSL 1        0.000007  0.001846   0.001846  0.001846   
                        3        0.000034  0.004106   0.004106  0.004106   
                        6        0.000081  0.006303   0.006303  0.006303   
               INDPRO   1        0.000099  0.005128   0.005128  0.005128   
                        3        0.000359  0.010254   0.010254  0.010254   
...                                   ...       ...        ...       ...   
xgboost  A     PAYEMS   3        0.000143  0.003763   0.003763  0.003763   
                        6        0.000186  0.005524   0.005524  0.005524   
               S&P 500  1        0.001455  0.027603   0.027603  0.027603   
                        3        0.004981  0.052461   0.052461  0.052461   
                        6        0.008941  0.071194   0.071194  0.071194   

                                     mase     rmspe  
model    track target   horizon                      
adaboost A     CPIAUCSL 1        0.974208  1.565714  
                        3        0.879556  2.461188  
                        6        0.905212  1.005820  
               INDPRO   1        0.821089  3.988761  
                        3        0.862412  4.544754  
...                                   ...       ...  
xgboost  A     PAYEMS   3        1.171336  1.406495  
                        6        0.966657  1.225171  
               S&P 500  1        0.837016  4.825886  
                        3        0.753449  2.059095  
                        6        0.701076  2.156708  

[120 rows x 6 columns]

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model   track           target   horizon usrec                                  
AR6     B               CPIAUCSL 1       0      0.000005  0.001586   0.001586   
                                         1      0.000019  0.003242   0.003242   
                                 3       0      0.000027  0.003785   0.003785   
                                         1      0.000161  0.009629   0.009629   
                                 6       0      0.000093  0.006961   0.006961   
...                                                  ...       ...        ...   
xgboost Q_zz_single_bit S&P 500  1       1      0.003793  0.046751   0.046751   
                                 3       0      0.004647  0.051856   0.051856   
                                         1      0.014571  0.097236   0.097236   
                                 6       0      0.009090  0.074541   0.074541   
                                         1      0.042596  0.170972   0.170972   

                                                    rmse      mase     rmspe  
model   track           target   horizon usrec                                
AR6     B               CPIAUCSL 1       0      0.001586  0.860288  1.348996  
                                         1      0.003242  1.546022  1.333686  
                                 3       0      0.003785  0.823602  2.077020  
                                         1      0.009629  1.925051  1.826281  
                                 6       0      0.006961  1.001412  1.211426  
...                                                  ...       ...       ...  
xgboost Q_zz_single_bit S&P 500  1       1      0.046751  1.326651  1.306572  
                                 3       0      0.051856  0.752689  2.227379  
                                         1      0.097236  1.368405  1.170910  
                                 6       0      0.074541  0.752026  2.015619  
                                         1      0.170972  1.648774  1.090794  

[3192 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,adaboost,A,CPIAUCSL,1,0,1972-04-01,0.002407,0.003220
1,adaboost,A,CPIAUCSL,1,1,1972-05-01,0.002401,0.002653
2,adaboost,A,CPIAUCSL,1,2,1972-06-01,0.002395,0.002630
3,adaboost,A,CPIAUCSL,1,3,1972-07-01,0.002389,0.002568
4,adaboost,A,CPIAUCSL,1,4,1972-08-01,0.004762,0.002906


,,mean,sum
model,track,,
adaboost,A,0.286,2207.821
decision_tree,A,0.023,175.838
gru,A,0.211,1628.746
linear_readout,A,0.017,130.482
linear_svm,A,0.076,584.631
lstm,A,0.146,1125.136
random_forest,A,0.227,1751.818
rnn,A,0.111,855.492
sig_kernel,A,0.053,408.023
